In [1]:
import pickle
import pandas as pd

In [2]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

In [3]:
air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

In [4]:
air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

In [5]:
dataset = pd.concat([weather, air_qual], axis=1)
dataset.drop(columns=["time"], inplace=True)

In [6]:
dataset

,dew_point_2m (°C),relative_humidity_2m (%),precipitation (mm),weather_code (wmo code),wind_speed_100m (km/h),wind_direction_10m (°),cloud_cover_high (%),cloud_cover_mid (%),cloud_cover_low (%),cloud_cover (%),surface_pressure (hPa),timestamp,pm10 (μg/m³),pm2_5 (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),carbon_monoxide (μg/m³),ozone (μg/m³)
0,1.5,88,0.0,0,22.3,233,0,0,0,0,1011.8,2016-01-01 00:00:00,12.7,5.9,22.0,3.0,188.0,34.0
1,1.0,91,0.0,0,16.8,210,14,0,0,14,1012.1,2016-01-01 01:00:00,16.5,8.6,19.0,4.1,198.0,26.0
2,0.1,92,0.0,0,15.9,189,1,0,0,1,1012.1,2016-01-01 02:00:00,21.0,12.6,20.4,4.4,202.0,27.0
3,-0.4,94,0.0,2,19.1,182,75,2,0,76,1012.8,2016-01-01 03:00:00,15.4,10.3,20.8,2.7,207.0,23.0
4,-0.5,94,0.0,3,19.7,170,100,90,0,100,1012.9,2016-01-01 04:00:00,15.7,9.1,23.0,2.2,164.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87667,-1.5,97,0.0,2,23.3,250,0,0,77,77,1017.5,2025-12-31 19:00:00,18.3,12.6,21.5,2.0,215.0,35.0
87668,-1.7,95,0.0,1,24.7,254,0,0,21,21,1016.9,2025-12-31 20:00:00,17.0,12.0,21.2,2.4,225.0,37.0
87669,-1.8,93,0.0,1,26.3,254,0,0,31,31,1015.9,2025-12-31 21:00:00,15.9,11.6,18.2,2.3,225.0,39.0
87670,-1.5,92,0.0,1,28.0,252,0,0,41,41,1014.6,2025-12-31 22:00:00,15.4,10.9,22.6,2.4,250.0,42.0


# rf base line with lag

In [7]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns

/data/ll2531/venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [8]:
air_qual_no_time = dataset.drop(columns=["timestamp"]).copy()

In [9]:
df = air_qual.copy()

# Create lag features
for col in [
    "pm10 (μg/m³)",
    "pm2_5 (μg/m³)",
    "nitrogen_dioxide (μg/m³)",
    "sulphur_dioxide (μg/m³)",
    "carbon_monoxide (μg/m³)",
    "ozone (μg/m³)"
]:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag3"] = df[col].shift(3)

# Target is current PM2.5
y = df["pm2_5 (μg/m³)"]

# Keep ONLY lagged features
x = df[[c for c in df.columns if "_lag" in c]]

# Remove rows with NaNs from shifting
mask = x.notna().all(axis=1)
x = x[mask]
y = y[mask]

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

rf = RandomForestRegressor(random_state=42)
rf.fit(x_train, y_train)

pred = rf.predict(x_test)

In [11]:
pred = rf.predict(x_test)
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("MAE", mae)
print("RMSE", np.sqrt(mean_squared_error(y_test, pred)))
print("R2", r2)

MAE 0.613991388160146
RMSE 0.9652411813103557
R2 0.9701230428704614


# lstm

In [12]:
# Separate features and target
features_df = dataset.drop(columns=["timestamp", "pm2_5 (μg/m³)"])
target_df = dataset[["pm2_5 (μg/m³)"]]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaled_X = scaler_X.fit_transform(features_df)
scaled_y = scaler_y.fit_transform(target_df)

lookback = 24*5
forecast_steps = 24 
# Construct inputs/outputs safely
x = []
y = []
for i in range(lookback, len(dataset) - forecast_steps + 1):
    x.append(scaled_X[i-lookback:i, :])
    y.append(scaled_y[i:i+forecast_steps, 0])

x = np.array(x)
y = np.array(y)

In [13]:
# Train/Test Split
split = int(len(x) * 0.7)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(128, input_shape=(x_train.shape[1], x_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    
    LSTM(64, return_sequences=False), 
    Dropout(0.2),
    
    Dense(32, activation="relu"),
    Dense(forecast_steps) 
])

model.compile(optimizer="adam", loss="mse")

model.fit(x_train, y_train, epochs=30, batch_size=64)

2026-06-07 21:42:39.984913: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-07 21:42:40.020158: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-07 21:42:40.944802: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:17808649

Epoch 1/30


2026-06-07 21:42:44.391493: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


958/958 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0040
Epoch 2/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0031
Epoch 3/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0029
Epoch 4/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0028
Epoch 5/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0026
Epoch 6/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0024
Epoch 7/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0022
Epoch 8/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step - loss: 0.0020
Epoch 9/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - loss: 0.0018
Epoch 10/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - loss: 0.0016
Epoch 11/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - loss: 0.0014
Epoch 12/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - loss: 0.0013
Epoch 13/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - loss: 0.0012
Epoch 14/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - loss: 0.0010
Epoch 15/30
958/958 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/ste

In [15]:
pred = model.predict(x_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

821/821 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step
MAE: 0.03975616327252006
RMSE: 0.057268617380815054
R²: 0.36185572742850125


In [16]:
import plotly.graph_objects as go


y_test_pred_rescaled = scaler_y.inverse_transform(pred)
y_test_true_rescaled = scaler_y.inverse_transform(y_test)

test_start_idx = split + lookback
pm25_raw = air_qual["pm2_5 (μg/m³)"].values

# Timeline Arrays for X-axis positioning
history_x = list(range(lookback))
forecast_x = list(range(lookback, lookback + forecast_steps))

fig = go.Figure()

sample_idx = 0
global_start = test_start_idx + sample_idx

fig.add_trace(go.Scatter(
    x=history_x, 
    y=pm25_raw[global_start - lookback : global_start],
    mode='lines+markers', name='Historical PM2.5', line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_true_rescaled[sample_idx],
    mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash')
))

fig.add_trace(go.Scatter(
    x=forecast_x, 
    y=y_test_pred_rescaled[sample_idx],
    mode='lines+markers', name='Predicted Forecast', line=dict(color='red')
))

# 2. Build the dropdown menus for the first 50 test samples
num_samples_to_show = min(50, len(x_test))
buttons = []

for idx in range(num_samples_to_show):
    visibility = [False] * (num_samples_to_show * 3)
    visibility[idx * 3] = True
    visibility[idx * 3 + 1] = True
    visibility[idx * 3 + 2] = True
    
    button = dict(
        label=f"Sample {idx}",
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"PM2.5 Timeline: 24h History + 5-Step Forecast (Sample {idx})"}
        ]
    )
    buttons.append(button)

# 3. Generate all remaining hidden traces for the selector upfront
for idx in range(1, num_samples_to_show):
    g_start = test_start_idx + idx
    fig.add_trace(go.Scatter(
        x=history_x, y=pm25_raw[g_start - lookback : g_start],
        mode='lines+markers', name='Historical PM2.5', line=dict(color='blue'), visible=False
    ))
    # Actual
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_true_rescaled[idx],
        mode='lines+markers', name='Actual Future', line=dict(color='green', dash='dash'), visible=False
    ))
    # Predicted
    fig.add_trace(go.Scatter(
        x=forecast_x, y=y_test_pred_rescaled[idx],
        mode='lines+markers', name='Predicted Forecast', line=dict(color='red'), visible=False
    ))

# 4. Final Layout Customization
fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons, direction="down", pad={"r": 10, "t": 10}, showactive=True, x=0.02, xanchor="left", y=1.15, yanchor="top")],
    title="PM2.5 Timeline: 24h History + 5-Step Forecast (Sample 0)",
    xaxis_title="Timeline (Hours)",
    yaxis_title="PM2.5 Concentration (μg/m³)",
    height=600,
    showlegend=True
)

fig.add_vline(x=23.5, line_width=2, line_dash="dash", line_color="gray")

fig.show()